# E83 五组模仿与好感数据的有限核对

## tl;dr
2026-11-05本地核对：处理后538行，五组235/76/81/72/74；七题均分逐行相符，alpha .936824。Kruskal–Wallis H=40.684947；DSCF模仿对静坐p约9.53e-8、对重复动作.008246、对自然随机动作.201206、对无动作指令组.628490。相对哪些对照成立须分别解释。

这是一份数据质量与显示值核对记录。编号列不唯一；原始555行与处理后538行不能简单按编号连接或只删17行重建。年龄范围/总体描述、论文模仿中位数和部分p值显示差异待解释。不将这些差异隐去，也不把复算成相近统计量称为原jamovi完整复现或真实关系有效。

## Context & Methods
来源：Trzmielewska等，*Examining the Impact of Control Condition Design in Mimicry–Liking Link Research*，[期刊正文与作者资料入口](https://czasopisma.tnkul.pl/index.php/rpsych/article/view/1665)，卷期2024、2025-05-14上线。实际读方法/结果/局限等指定网页PDF文本，未取得本地PDF或逐页图像。数据依据论文所给共享资料入口本地取得；不在笔记中嵌入共享访问参数、原始数据、个人编号或自由文本。

### Key Assumptions
每行按一条问卷记录处理；不能凭非唯一lp认证独立个人或助手。未见助手ID，未校正同助手聚类；无动作指令组由另外两助手采集，不能当五组完全同样随机化。七题等权均值，不读取purposeofthestudy/study_purpose内容；不手工改年龄或结果。DSCF逐对重排秩、校正同分后以五组studentized range求p，参考[原实现及方法说明](https://scikit-posthocs.readthedocs.io/en/latest/generated/scikit_posthocs.posthoc_dscf.html)，未运行原jamovi。

## Data
本笔记不打包SAV。将原始文件按如下名放同目录：e83_control_raw.sav、e83_control_processed.sav；设置MIMICRY_INPUT_DIR到该目录。两指纹在首单元校验；版本不同时停止而非套旧结果。依赖Python、numpy、pandas、scipy、pyreadstat；安装依赖与取得输入后，从新内核顺序运行全部四代码单元。仓库不自动下载数据或联系作者。

### 1. 载入并核对输入版本


In [1]:
from pathlib import Path
import sys, os, json, hashlib
from collections import Counter
here = Path.cwd()
candidates = ([Path(os.environ['MIMICRY_INPUT_DIR'])] if os.environ.get('MIMICRY_INPUT_DIR') else [])
candidates += [here, here/'tmp'] + [parent/'tmp' for parent in here.parents]
data_dir = next((p for p in candidates if (p/'e83_control_processed.sav').is_file() and (p/'e83_control_raw.sav').is_file()), None)
if data_dir is None:
    raise FileNotFoundError('请将两个指定SAV放在同一目录，并设置MIMICRY_INPUT_DIR；来源和指纹见说明。')
for name in ['gaze_deps','e30_deps']:
    local = data_dir/name
    if local.is_dir(): sys.path.insert(0,str(local))
import numpy as np, pandas as pd, pyreadstat, scipy
from scipy import stats
p, pm = pyreadstat.read_sav(data_dir/'e83_control_processed.sav')
r, rm = pyreadstat.read_sav(data_dir/'e83_control_raw.sav')
liking = [f'L{i}' for i in range(1,8)]
expected = {'processed':'be1dd8cbd8e632c31ed2ffd34d0e46606aee96fe818f80cd77377e03a9efc7de',
            'raw':'a371c91ab1d52adabe8615b8db1494ec7ebe87eecd6990c3052f8c3e97565223'}
actual = {name:hashlib.sha256((data_dir/f'e83_control_{name}.sav').read_bytes()).hexdigest() for name in expected}
if actual != expected: raise ValueError('输入版本指纹与本记录不同，须重新核查版本。')
print(json.dumps({'hashes_match':True,'processed_shape':list(p.shape),'raw_shape':list(r.shape),
                  'numpy':np.__version__,'pandas':pd.__version__,'scipy':scipy.__version__},ensure_ascii=False,indent=2))


{
  "hashes_match": true,
  "processed_shape": [
    538,
    26
  ],
  "raw_shape": [
    555,
    23
  ],
  "numpy": "2.3.5",
  "pandas": "3.0.1",
  "scipy": "1.18.1"
}


## Results

### 2. 数据结构、编号与两版本对应
仅输出汇总。精确数值多重集比对不靠非唯一编号连接。第二种问卷响应签名有重复，故只比较唯一匹配项，并明确它不证明身份；不能据34条差异记录直接推断新增34人。


In [2]:
profile={}
for name,d in [('processed',p),('raw',r)]:
 profile[name]={'rows':len(d),'columns':len(d.columns),'key_unique':bool(d.lp.is_unique),'key_missing':int(d.lp.isna().sum()),'condition_counts':{int(k):int(v) for k,v in d.condition.value_counts().sort_index().items()},'gender_counts':{int(k):int(v) for k,v in d.gender.value_counts().sort_index().items()},'age':{'n':int(d.age.notna().sum()),'mean':float(d.age.mean()),'sd':float(d.age.std()),'min':float(d.age.min()),'max':float(d.age.max())},'liking_missing_cells':int(d[liking].isna().sum().sum()),'liking_invalid_cells':int(((d[liking]<1)|(d[liking]>7)).sum().sum()),'missing_affect_cells':int(d[[f'N{i}' for i in range(1,11)]].isna().sum().sum())}
out={'source_hashes':{name:hashlib.sha256((data_dir/f'e83_control_{name}.sav').read_bytes()).hexdigest() for name in ['processed','raw']},'profile':profile}
common=['lp','condition','gender','age']+liking+[f'N{i}' for i in range(1,11)]
def sigs(d):
 return Counter(tuple(None if pd.isna(x) else float(x) for x in row) for row in d[common].itertuples(index=False,name=None))
pc,rc=sigs(p),sigs(r)
extra,excluded=pc-rc,rc-pc
out['filter_reconciliation']={'method':'Exact multiset of selected numeric fields including nonunique lp; not a person-identity join; free text excluded','processed_numeric_records_not_raw':sum(extra.values()),'raw_numeric_records_not_processed':sum(excluded.values()),'excluded_condition_counts':{int(c):sum(n for row,n in excluded.items() if row[1]==c) for c in sorted(r.condition.unique())},'unique_lp_processed':int(p.lp.nunique()),'unique_lp_raw':int(r.lp.nunique()),'duplicate_lp_rows_processed':int(p.lp.duplicated(keep=False).sum()),'duplicate_lp_rows_raw':int(r.lp.duplicated(keep=False).sum()),'exact_numeric_duplicate_rows_processed':sum(n for n in pc.values() if n>1),'exact_numeric_duplicate_rows_raw':sum(n for n in rc.values() if n>1)}
stable=['condition','gender']+liking+[f'N{i}' for i in range(1,11)]
def records(d):
 out={};duplicates=0
 for _,row in d.iterrows():
  key=tuple(None if pd.isna(row[c]) else float(row[c]) for c in stable)
  if key in out:duplicates+=1
  out.setdefault(key,[]).append((float(row.lp),float(row.age)))
 return out,duplicates
pr,pdups=records(p);rr,rdups=records(r)
matched=[k for k in pr.keys()&rr.keys() if len(pr[k])==len(rr[k])==1]
out['signature_reconciliation']={'method':'Secondary exact condition+gender+17item numeric signature; not identity proof','duplicate_signatures_processed':pdups,'duplicate_signatures_raw':rdups,'unambiguous_matching_records':len(matched),'processed_signatures_not_raw':len(pr.keys()-rr.keys()),'raw_signatures_not_processed':len(rr.keys()-pr.keys()),'lp_differences_on_unambiguous_matches':sum(pr[k][0][0]!=rr[k][0][0] for k in matched),'age_differences_on_unambiguous_matches':sum(pr[k][0][1]!=rr[k][0][1] for k in matched),'lp_or_age_differences_on_unambiguous_matches':sum(pr[k][0]!=rr[k][0] for k in matched)}

print(json.dumps({k:out[k] for k in ['profile','filter_reconciliation','signature_reconciliation']},ensure_ascii=False,indent=2))


{
  "profile": {
    "processed": {
      "rows": 538,
      "columns": 26,
      "key_unique": false,
      "key_missing": 0,
      "condition_counts": {
        "1": 235,
        "2": 76,
        "3": 81,
        "4": 72,
        "5": 74
      },
      "gender_counts": {
        "1": 301,
        "2": 232,
        "3": 5
      },
      "age": {
        "n": 538,
        "mean": 25.44609665427509,
        "sd": 7.986130528917305,
        "min": 17.0,
        "max": 75.0
      },
      "liking_missing_cells": 0,
      "liking_invalid_cells": 0,
      "missing_affect_cells": 13
    },
    "raw": {
      "rows": 555,
      "columns": 23,
      "key_unique": false,
      "key_missing": 0,
      "condition_counts": {
        "1": 241,
        "2": 80,
        "3": 81,
        "4": 79,
        "5": 74
      },
      "gender_counts": {
        "1": 309,
        "2": 241,
        "3": 5
      },
      "age": {
        "n": 555,
        "mean": 25.418018018018017,
        "sd": 7.9123150162668

### 3. 计分、描述与总检验
表中分位数采用pandas线性插值，原软件分位数定义不明；差异不直接判为数据错误。整数总分与均分的秩检验另作相符核对。


In [3]:
computed=p[liking].mean(axis=1)
out['score_max_abs_diff']=float(np.max(np.abs(computed-p.liking_mean)))
out['cronbach_alpha']=float(7/6*(1-p[liking].var(ddof=1).sum()/p[liking].sum(axis=1).var(ddof=1)))
ranks=stats.rankdata(p.liking_mean)
groups=[]
for c,g in p.groupby('condition'):
 vals=g.liking_mean
 groups.append({'condition':int(c),'n':len(g),'mean':float(vals.mean()),'median':float(vals.median()),'min':float(vals.min()),'max':float(vals.max()),'q1_linear':float(vals.quantile(.25)),'q3_linear':float(vals.quantile(.75)),'mean_rank':float(np.mean(ranks[p.condition==c]))})
out['groups']=groups
test=stats.kruskal(*[g.liking_mean.to_numpy() for _,g in p.groupby('condition')])
out['kruskal_wallis']={'H':float(test.statistic),'df':4,'p':float(test.pvalue),'epsilon_squared_H_over_nminus1':float(test.statistic/(len(p)-1)),'eta_squared_HminusKplus1_over_nminusK':float((test.statistic-4)/(len(p)-5))}
out['age_by_gender']=[{'gender':int(c),'n':len(g),'mean':float(g.age.mean()),'sd':float(g.age.std())} for c,g in p.groupby('gender')]
out['canonical_sum_check']={'distinct_mean_scores':int(p.liking_mean.nunique()),'distinct_sum_scores':int(p[liking].sum(axis=1).nunique()),'H_from_integer_sums':float(stats.kruskal(*[g[liking].sum(axis=1).to_numpy() for _,g in p.groupby('condition')]).statistic)}

print(pd.DataFrame(out['groups']).round(5).to_string(index=False))
print(json.dumps({k:out[k] for k in ['score_max_abs_diff','cronbach_alpha','kruskal_wallis','age_by_gender','canonical_sum_check']},ensure_ascii=False,indent=2))


 condition   n    mean  median     min  max  q1_linear  q3_linear  mean_rank
         1 235 5.83951 5.85714 2.85714  7.0    5.14286    6.71429  305.78298
         2  76 4.68797 4.78571 1.57143  7.0    3.53571    5.89286  181.86842
         3  81 5.56261 5.71429 3.14286  7.0    5.00000    6.28571  265.47531
         4  72 5.33730 5.42857 2.57143  7.0    4.67857    6.17857  236.93056
         5  74 5.69112 5.71429 3.71429  7.0    5.17857    6.14286  280.37162
{
  "score_max_abs_diff": 0.0,
  "cronbach_alpha": 0.9368238498607551,
  "kruskal_wallis": {
    "H": 40.68494655359051,
    "df": 4,
    "p": 3.123351440486752e-08,
    "epsilon_squared_H_over_nminus1": 0.07576340140333428,
    "eta_squared_HminusKplus1_over_nminusK": 0.06882729184538557
  },
  "age_by_gender": [
    {
      "gender": 1,
      "n": 301,
      "mean": 25.98671096345515,
      "sd": 8.865278120069297
    },
    {
      "gender": 2,
      "n": 232,
      "mean": 24.82758620689655,
      "sd": 6.69725777321195
    },
 

### 4. 全部十组比较及聚类限制
W列固定为较小U的方向，绝对值用于检验，不是原组别差异方向。p值已按五组studentized range处理，不再附加一次Bonferroni。相同计算是可检验核对，不是对所有原分析、排除和实际随机化的认证。


In [4]:
dscf=[]
for i in range(1,6):
 for j in range(i+1,6):
  left=p.loc[p.condition==i,'liking_mean'].to_numpy();right=p.loc[p.condition==j,'liking_mean'].to_numpy()
  ni,nj=len(left),len(right);n=ni+nj
  pranks=stats.rankdata(np.concatenate([left,right]));u=pranks[:ni].sum()-ni*(ni+1)/2
  counts=np.unique(pranks,return_counts=True)[1]
  correction=np.sum(counts**3-counts)/12
  variance=ni*nj/(n*(n-1))*((n**3-n)/12-correction)
  q=np.sqrt(2)*(min(u,ni*nj-u)-ni*nj/2)/np.sqrt(variance)
  dscf.append({'left':i,'right':j,'W_unsigned_direction':float(q),'p_all5_studentized_range':float(stats.studentized_range.sf(abs(q),5,np.inf))})
out['DSCF']=dscf
out['cluster_limit']={'confederate_identifier_present':any('confed' in c.lower() or 'interview' in c.lower() for c in p.columns),'note':'No direct confederate identifier found among column names; not proof no assignment existed elsewhere. Free-text study-purpose responses not inspected or exported.'}
out['limits']='Local descriptive/score/Kruskal-Wallis and DSCF reconciliation. DSCF uses within-pair reranks/tie correction and studentized range across5, inspected against primary scikit-posthocs source; not original jamovi execution. No individual identifiers, free text or rows exported; no cluster adjustment, causal or long-term validation.'

print(pd.DataFrame(out['DSCF']).round(8).to_string(index=False))
print(json.dumps({'cluster_limit':out['cluster_limit'],'limits':out['limits']},ensure_ascii=False,indent=2))


 left  right  W_unsigned_direction  p_all5_studentized_range
    1      2             -8.115627              1.000000e-07
    1      3             -3.033114              2.012056e-01
    1      4             -4.683358              8.245550e-03
    1      5             -1.978082              6.284896e-01
    2      3             -5.084490              2.990930e-03
    2      4             -3.527658              9.183027e-02
    2      5             -5.790754              4.067600e-04
    3      4             -1.757727              7.261218e-01
    3      5             -0.870657              9.726621e-01
    4      5             -2.659150              3.279354e-01


{
  "cluster_limit": {
    "confederate_identifier_present": false,
    "note": "No direct confederate identifier found among column names; not proof no assignment existed elsewhere. Free-text study-purpose responses not inspected or exported."
  },
  "limits": "Local descriptive/score/Kruskal-Wallis and DSCF reconciliation. DSCF uses within-pair reranks/tie correction and studentized range across5, inspected against primary scikit-posthocs source; not original jamovi execution. No individual identifiers, free text or rows exported; no cluster adjustment, causal or long-term validation."
}


## Takeaways
本地结果支持所给版本的计分与若干显示值；模仿对自然动作/无动作指令的比较仍不显著，但这不构成严格等效结论。原数据编号不唯一、年龄/处理转换不明和助手聚类缺口限制更强结论。对照安排与后测好感不能外推成长期亲密必要条件。

验证记录：nbformat正式格式及四单元新内核顺序执行单列留痕。Notebook完整界面/HTML阅读效果尚未视觉验收；不能以执行成功代替界面验收。结果表采用有界文本输出，不展示个人记录。旧五份笔记本轮未改未重跑。
